<a href="https://colab.research.google.com/github/yanapatt/Mushrooms-Classification/blob/main/%5BD%5D_MushroomCommonGenus_MCG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Mushroom Common Genus's Dataset**
---
Source: https://www.kaggle.com/datasets/maysee/mushrooms-classification-common-genuss-images

In [31]:
import os
from dotenv import load_dotenv
import pathlib
import shutil
import zipfile

from kaggle.api.kaggle_api_extended import KaggleApi

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

import numpy as np
import random as rd

from sklearn.model_selection import train_test_split
from tqdm import tqdm

## **Section 1: Setting directory**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
drive_dir = "/content/drive/MyDrive/projectMushroom" # ตำแหน่ง Root Project Folder ใน G-Drive
dataset_drive_dir = f"{drive_dir}/data/mushroomCommonGenus"
os.chdir(drive_dir)
print(f"Current directory: {os.getcwd()}")

Current directory: /content/drive/MyDrive/projectMushroom


In [8]:
load_dotenv()
os.environ.get("KAGGLE_USERNAME")
os.environ.get("KAGGLE_KEY")

api = KaggleApi()
api.authenticate()

In [16]:
runtime_dir = "/content"
dataset_dir = os.path.join(runtime_dir, "mcg_dataset") # Dataset ปกติ
dataset_augA_dir = os.path.join(runtime_dir, "mcg_dataset_a") # Augment ก่อนแบ่ง Dataset
dataset_augB_dir = os.path.join(runtime_dir, "mcg_dataset_b") # แบ่ง Dataset ก่อน Augment

os.makedirs(dataset_dir, exist_ok=True)
os.makedirs(dataset_augA_dir, exist_ok=True)
os.makedirs(dataset_augB_dir, exist_ok=True)

**Download dataset from kaggle**

In [45]:
src_dataset = "maysee/mushrooms-classification-common-genuss-images" # Dataset จาก kaggle
!kaggle datasets download -d {src_dataset} -p {runtime_dir} --unzip # Download dataset and Extract to output path

Dataset URL: https://www.kaggle.com/datasets/maysee/mushrooms-classification-common-genuss-images
License(s): copyright-authors
 99% 1.84G/1.85G [00:32<00:00, 125MB/s] 
100% 1.85G/1.85G [00:32<00:00, 61.1MB/s]


In [46]:
shutil.rmtree("/content/mushrooms") # ลบ Folder ที่ไม่จำเป็น

## **Section 2: Data cleaning**

**Detect and Remove incorrupted images**

In [47]:
raw_data_dir = "/content/Mushrooms"

In [48]:
invalid_df = pd.read_csv(f"{dataset_drive_dir}/invalid_idx.csv")
invalid_df["Filepath"] = invalid_df["Filepath"].str.replace("/content/dataset", "/content/Mushrooms")
invalid_df

,Filepath,Classes
0,/content/Mushrooms/Agaricus/366_XEI1sAQds7w.jpg,Agaricus
1,/content/Mushrooms/Agaricus/273_ZwxysL6gjVQ.jpg,Agaricus
2,/content/Mushrooms/Boletus/0572_zdESww0j-to.jpg,Boletus
3,/content/Mushrooms/Boletus/0726_wNJGG_syL80.jpg,Boletus
4,/content/Mushrooms/Boletus/0507_s5LwdPfx9sU.jpg,Boletus
5,/content/Mushrooms/Boletus/0051_rBIC-Uy9KzI.jpg,Boletus
6,/content/Mushrooms/Boletus/0105_7lVm0lLO6FQ.jpg,Boletus
7,/content/Mushrooms/Boletus/0573_O4Ftoobc-mM.jpg,Boletus
8,/content/Mushrooms/Lactarius/0962_bCxHyrNZhYE.jpg,Lactarius
9,/content/Mushrooms/Lactarius/127_A77QQ1QPZ1E.jpg,Lactarius


In [56]:
def detect_and_remove_images(root_dir, invalid_paths=None, remove=False, extensions=(".jpg", ".jpeg", ".png")):
    ImageFile.LOAD_TRUNCATED_IMAGES = False
    corrupted_images = set()

    root_dir = pathlib.Path(root_dir)

    if invalid_paths:
        for p in invalid_paths:
            try:
                corrupted_images.add(pathlib.Path(p))
            except Exception as e:
                print(f"[Log]: Error adding {p}: {e}")
                pass

    image_files = [
        p for p in root_dir.rglob("*")
        if p.suffix.lower() in extensions
    ]

    for img_path in tqdm(image_files, desc="Checking images"):
        try:
            with Image.open(img_path) as img:
                img.load()
        except Exception as e:
            print(f"\n[Log]: Corrupted image: {img_path}, Error: {e}")
            corrupted_images.add(img_path)

    print(f"Total corrupted images: {len(corrupted_images)}")

    if remove:
        for img_path in corrupted_images:
            try:
                if img_path.exists():
                    print(f"[Log]: Removing {img_path}")
                    os.remove(img_path)
            except Exception as e:
                print(f"[Log]: Error removing {img_path}: {e}")
                pass
        print("Complete!")

    return list(corrupted_images)

In [57]:
img_corrupted = invalid_df["Filepath"].tolist()
bad_images = detect_and_remove_images(root_dir=raw_data_dir, invalid_paths=img_corrupted, remove=True)

Checking images:  32%|███▏      | 2150/6714 [00:09<00:16, 278.15it/s]


[Log]: Corrupted image: /content/Mushrooms/Russula/092_43B354vYxm8.jpg, Error: image file is truncated (92 bytes not processed)


Checking images: 100%|██████████| 6714/6714 [00:30<00:00, 223.26it/s]

Total corrupted images: 19
[Log]: Removing /content/Mushrooms/Boletus/0105_7lVm0lLO6FQ.jpg
[Log]: Removing /content/Mushrooms/Russula/426_RHi0dlqqt8o.jpg
[Log]: Removing /content/Mushrooms/Boletus/0507_s5LwdPfx9sU.jpg
[Log]: Removing /content/Mushrooms/Lactarius/0962_bCxHyrNZhYE.jpg
[Log]: Removing /content/Mushrooms/Russula/631_G3z48A44RsE.jpg
[Log]: Removing /content/Mushrooms/Boletus/0573_O4Ftoobc-mM.jpg
[Log]: Removing /content/Mushrooms/Agaricus/366_XEI1sAQds7w.jpg
[Log]: Removing /content/Mushrooms/Lactarius/0359_GY1BVriI5RQ.jpg
[Log]: Removing /content/Mushrooms/Boletus/0051_rBIC-Uy9KzI.jpg
[Log]: Removing /content/Mushrooms/Boletus/0726_wNJGG_syL80.jpg
[Log]: Removing /content/Mushrooms/Boletus/0572_zdESww0j-to.jpg
[Log]: Removing /content/Mushrooms/Russula/092_43B354vYxm8.jpg
[Log]: Removing /content/Mushrooms/Russula/151_yX7oI-mM9hk.jpg
[Log]: Removing /content/Mushrooms/Russula/043_a_5lm55zCVc.jpg
[Log]: Removing /content/Mushrooms/Lactarius/127_A77QQ1QPZ1E.jpg
[Log]: Removi